# SeqMixer on Colab: MLA vs MHA vs SWA vs Mamba

Run the controlled recall–memory–compute study on a **GPU**.

1. Set the runtime to a GPU: **Runtime → Change runtime type → GPU**.
2. Run the cells top to bottom.

On a GPU you can use the larger `--scale full` configs; on CPU stick to `--scale demo`.

## 1. Get the code

Edit `REPO_URL` / `BRANCH` if you push the project to your own repo (e.g. `gmaharsh/seqmix`).

In [ ]:
REPO_URL = "https://github.com/gmaharsh/2048.git"   # or https://github.com/gmaharsh/seqmix.git
BRANCH   = "cursor/seqmixer-pareto-study-47f5"        # use "main" for the seqmix repo

import os
if not os.path.exists('seqmix_repo'):
    !git clone --depth 1 --branch $BRANCH $REPO_URL seqmix_repo
%cd seqmix_repo
!ls

## 2. Install dependencies

Colab already ships PyTorch with CUDA. We just need numpy + matplotlib.

Optionally install the fast Mamba CUDA kernels (the repo's reference Mamba runs without them, just slower).

In [ ]:
!pip -q install numpy matplotlib
import torch
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Optional (GPU only): fast Mamba kernels. Safe to skip.
# !pip -q install causal-conv1d mamba-ssm

## 3. Smoke test

Confirm the four mixers build and the incremental decode matches the parallel forward.

In [ ]:
import sys; sys.path.insert(0, 'src')
import torch
from seqmix.config import ModelConfig
from seqmix.model import LanguageModel
from seqmix.utils import count_non_embedding_parameters

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
for mixer, kw in [('mha', {}), ('swa', {'window': 16}), ('mla', {'d_c': 64}), ('mamba', {'d_state': 16})]:
    cfg = ModelConfig(vocab_size=64, max_seq_len=64, d_model=128, n_layers=2, n_heads=4, mixer=mixer, mixer_kwargs=kw)
    m = LanguageModel(cfg).to(dev)
    idx = torch.randint(0, 64, (2, 64), device=dev)
    logits, loss = m(idx, idx)
    print(f"{mixer:6s} loss={loss.item():.3f} params={count_non_embedding_parameters(m)} cache@64={m.analytic_state_bytes(64)}B")

## 4. Run the phases

Use `--scale full --device cuda` on a GPU; `--scale demo` is the quick CPU-sized version. Adjust as you like.

In [ ]:
SCALE = 'full' if torch.cuda.is_available() else 'demo'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('scale', SCALE, 'device', DEVICE)

# Phase 1: recall-memory frontier (the headline figure)
!PYTHONPATH=src python experiments/run_synthetic.py --task mqar --scale $SCALE --device $DEVICE

In [ ]:
# Phase 2: language modeling (synthetic recall corpus and/or TinyStories)
!PYTHONPATH=src python experiments/run_lm.py --scale $SCALE --device $DEVICE
!PYTHONPATH=src python experiments/run_lm.py --scale $SCALE --device $DEVICE --dataset tinystories

In [ ]:
# Compute/memory profiling + long-context + mechanistic
!PYTHONPATH=src python experiments/run_efficiency.py --scale $SCALE --device $DEVICE
!PYTHONPATH=src python experiments/run_longctx.py   --scale $SCALE --device $DEVICE
!PYTHONPATH=src python experiments/run_mechanistic.py --scale $SCALE --device $DEVICE

## 5. Figures

In [ ]:
!PYTHONPATH=src python analysis/plot.py $SCALE
!PYTHONPATH=src python analysis/summarize.py $SCALE

import glob
from IPython.display import Image, display
for fig in sorted(glob.glob('paper/figures/*' + SCALE + '*.png')):
    print(fig)
    display(Image(fig))